## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
from scipy.interpolate import RegularGridInterpolator

## 2. Functions

In [2]:
def get_numbers(arr):
    numbers = []

    for item in arr:
        match = re.search(r'-?\d+\.?\d*', item)
        if match:
            numbers.append(float(match.group()))

    return np.array(numbers)
    

## 3. THS Data Preparation

In [3]:
ths_flaps_1f = pd.read_csv("../data/raw/ths_flaps_1f.csv")
ths_flaps_2 = pd.read_csv("../data/raw/ths_flaps_2.csv")

dataframes = {"ths_flaps_1f": ths_flaps_1f, "ths_flaps_2": ths_flaps_2}

for name, df in dataframes.items():
    weights = df["WeightT"].to_numpy(dtype=float)
    raw_cgs = df.columns.to_list()
    cgs = get_numbers(raw_cgs)
    ths = df.iloc[:, 1:].to_numpy(dtype=float)

    interpolator = RegularGridInterpolator(
        (weights, cgs),
        ths
    )

    weights_values = np.round(
        np.arange(weights.min(), weights.max() + 0.1, 0.1),
        1
    )

    cgs_values = np.round(
        np.arange(cgs.min(), cgs.max() + 0.1, 0.1),
        1
    )

    weights_grid, cgs_grid = np.meshgrid(weights_values, cgs_values, indexing="ij")
    points = np.column_stack([
        weights_grid.ravel(),
        cgs_grid.ravel()
    ])

    ths_values = interpolator(points)
    ths_grid = ths_values.reshape(weights_grid.shape)

    final_df = pd.DataFrame(
        ths_grid,
        columns=[f"CG{cg:.1f}" for cg in cgs_values]
    )

    final_df.insert(0, "WeightT", weights_values)

    final_df.to_csv(f"../data/prepared/{name}_prepared.csv")

## 4. V1, VR, V2 Data Preparation

In [ ]:
speed_1f_0ft = pd.read_csv("../data/raw/speed_1f_0ft.csv")
speed_1f_1000ft = pd.read_csv("../data/raw/speed_1f_1000ft.csv")
speed_1f_2000ft = pd.read_csv("../data/raw/speed_1f_2000ft.csv")

speed_2_0ft = pd.read_csv("../data/raw/speed_2_0ft.csv")
speed_2_1000ft = pd.read_csv("../data/raw/speed_2_1000ft.csv")
speed_2_2000ft = pd.read_csv("../data/raw/speed_2_2000ft.csv")

dataframes = {
    "speed_1f_0ft": speed_1f_0ft, 
    "speed_1f_1000ft": speed_1f_1000ft, 
    "speed_1f_2000ft": speed_1f_2000ft,
    "speed_2_0ft": speed_2_0ft,
    "speed_2_1000ft": speed_2_1000ft,
    "speed_2_2000ft": speed_2_2000ft
}

value_columns = [
    "MaxTakeoffWeightT",
    "V1Kt",
    "VRKt",
    "V2Kt",
    "LimitationCode",
]

for name, df in dataframes.items():
    df = df.sort_values(
        ["CorrectedRunwayLengthM", "OATC"]
    ).reset_index(drop=True)

    result_rows = []

    for runway_length, group in df.groupby("CorrectedRunwayLengthM"):
        group = group.sort_values("OATC").reset_index(drop=True)

        oat = group["OATC"].to_numpy(dtype=float)

        oat_values = np.arange(
            int(oat.min()),
            int(oat.max()) + 1,
            1,
            dtype=float
        )

        interpolated_group = pd.DataFrame({
            "OATC": oat_values,
            "CorrectedRunwayLengthM": runway_length
        })

        for column in value_columns:
            y = group[column].to_numpy(dtype=float)

            interpolator = RegularGridInterpolator(
                (oat,),
                y
            )

            values = interpolator(oat_values.reshape(-1, 1))

            if column in ["V1Kt", "VRKt", "V2Kt", "LimitationCode"]:
                values = np.rint(values).astype(int)

            elif column == "MaxTakeoffWeightT":
                values = np.round(values, 2)

            interpolated_group[column] = values

        result_rows.append(interpolated_group)

    final_df = pd.concat(result_rows, ignore_index=True)

    final_df = final_df.sort_values(
        ["OATC", "CorrectedRunwayLengthM"]
    ).reset_index(drop=True)

    final_df.to_csv(
        f"../data/prepared/{name}_prepared.csv",
        index=False
    )